<a href="https://colab.research.google.com/github/lourrosaa/TP-Grupal-Analisis/blob/lour-branch/TP_analisis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
from google.colab import files
subido = files.upload()
print(subido.keys())

TypeError: 'NoneType' object is not subscriptable

Cargamos datos, importamos las librerias necesarias para el analisis y la carga del arhivo CSV. Mostrando la informacion principal del Dataframe con las primeras y ultimas filas, cantidad de filas y columnas, datos vacios y filas repetidas. Printeamos cada cosa para que no ejecute solo el ultimo codigo que le pedimos.

In [27]:
import pandas as pd
import numpy as np

import pandas as pd
df = pd.read_csv("sephora_website_dataset (1).csv")
df.head()
print(df.head())
df.tail() #para las ultimas
print(df.tail())
df.shape #cuenta filas y columnas
print(df.shape)
df.dtypes #tipo de dato por columna
print(df.dtypes)
df.isnull().sum() #filas repetidas
print(df.isnull().sum())

        id           brand   category                            name  \
0  2218774  Acqua Di Parma  Fragrance  Blu Mediterraneo MINIATURE Set   
1  2044816  Acqua Di Parma    Cologne                         Colonia   
2  1417567  Acqua Di Parma    Perfume                Arancia di Capri   
3  1417617  Acqua Di Parma    Perfume                Mirto di Panarea   
4  2218766  Acqua Di Parma  Fragrance           Colonia Miniature Set   

             size  rating  number_of_reviews  love  price  value_price  ...  \
0  5 x 0.16oz/5mL     4.0                  4  3002   66.0         75.0  ...   
1   0.7 oz/ 20 mL     4.5                 76  2700   66.0         66.0  ...   
2    5 oz/ 148 mL     4.5                 26  2600  180.0        180.0  ...   
3   2.5 oz/ 74 mL     4.5                 23  2900  120.0        120.0  ...   
4  5 x 0.16oz/5mL     3.5                  2   943   72.0         80.0  ...   

  MarketingFlags  MarketingFlags_content  \
0           True             online only  

Podemos ver que en este Dataset que importamos de Sephora ningun dato se encuentra vacio ademas de que la mayoria de datos son de texto o de numeros enteros, con solo pocas excepciones.

In [28]:
df[["limited_edition", "rating", "love"]].describe()
print(df[["limited_edition", "rating", "love"]].describe())
df["limited_edition"].value_counts()

       limited_edition       rating          love
count      9168.000000  9168.000000  9.168000e+03
mean          0.091841     3.990020  1.627859e+04
std           0.288817     1.007707  4.260651e+04
min           0.000000     0.000000  0.000000e+00
25%           0.000000     4.000000  1.600000e+03
50%           0.000000     4.000000  4.800000e+03
75%           0.000000     4.500000  1.380000e+04
max           1.000000     5.000000  1.300000e+06


,count
limited_edition,
0,8326
1,842


De los 9168 productos, solo 842 son de edicion limitada

In [29]:
df.groupby("limited_edition")["love"].median()


,love
limited_edition,
0,5000.0
1,3000.0


De los productos en total, se ve que los normales gustan mas que los productos de edicion limitada. Sin embargo tal vez eso no sea porque sean menos queridos por los compradores, se podria deber tambien a que al ser ediciones limitadas y durar menos tiempo en el mercado tienen menos tiempo para acumular "loves", no todos alcanzan a comprarlos o simplemente las personas prefieren los productos clasicos.

In [30]:
df_rating = df[df["rating"] > 0]
print(df_rating)
df_rating.groupby("limited_edition")["rating"].median()
print(df_rating.groupby("limited_edition")["rating"].median())
df_rating.groupby("limited_edition")["rating"].mean()
print(df_rating.groupby("limited_edition")["rating"].mean())

           id               brand                  category  \
0     2218774      Acqua Di Parma                 Fragrance   
1     2044816      Acqua Di Parma                   Cologne   
2     1417567      Acqua Di Parma                   Perfume   
3     1417617      Acqua Di Parma                   Perfume   
4     2218766      Acqua Di Parma                 Fragrance   
...       ...                 ...                       ...   
9160  2149847  SEPHORA COLLECTION  Bath Soaks & Bubble Bath   
9161  1601111  SEPHORA COLLECTION                Brush Sets   
9162  2149839  SEPHORA COLLECTION        Scrub & Exfoliants   
9163  2208502  SEPHORA COLLECTION                Face Masks   
9166       50  SEPHORA COLLECTION               no category   

                                                   name            size  \
0                        Blu Mediterraneo MINIATURE Set  5 x 0.16oz/5mL   
1                                               Colonia   0.7 oz/ 20 mL   
2                 

Aca vemos que no solo las ediciones limitadas tienen menos love que los productos normales, sino que tiene el mismo promedio de rating. Asi que por ahora no vemos que las ediciones limitadas gusten mas que las regulares. Despues lo confirmamos con los numeros exactos que da entre cada cosa.


## Pregunta de investigación

¿Los productos de edición limitada de Sephora tienen mejores ratings
y más "loves" que los productos regulares?

**¿Por qué vale la pena responderla?**
Las ediciones limitadas se promocionan como productos especiales.
Esto permite ver si esa estrategia se refleja en la calidad percibida
y en la popularidad entre los usuarios.


## Transformación 1:
limpieza de valores sin datos reales ya que algunas columnas usan textos como "unknown" o "no category" para indicar que falta información. Se reemplazan por valores vacíos (NaN) para que no se interpreten como datos reales.

**Asistencia de IA (Claude):** la idea de esta transformación surgió de
una consulta a Claude. Prompt utilizado: "Busca a ver si hay inconsistencias o datos que tal vez esten vacios y no aparezcan como null pero que no sirvan y explicame si me sirve partir de eso para hacer una transformacion".
La IA detectó los textos "unknown", "no category" y "no size" en una tabla
que le compartió. Se verificó su presencia en el dataset
con value_counts() y explicó parte del código ya que hubo problemas con la deteccion de las inconsistencias del dataset, explico los prints y el uso del "df original" ya que se hayaron problemas iniciales en cuanto a la verificacion.

In [37]:
df_original = pd.read_csv("sephora_website_dataset.csv")

print("ingredients = unknown:", (df_original["ingredients"] == "unknown").sum())
print("category = no category:", (df_original["category"] == "no category").sum())
print("size = no size:", (df_original["size"] == "no size").sum())

df["ingredients"] = df["ingredients"].replace("unknown", np.nan)
df["category"] = df["category"].replace("no category", np.nan)
df["size"] = df["size"].replace("no size", np.nan)

df[["ingredients", "category", "size"]].isnull().sum()

ingredients = unknown: 1630
category = no category: 16
size = no size: 3161


,0
ingredients,1630
category,16
size,3161


Tras reemplazar los textos “unknown”, “no category” y “no size” por valores vacíos, se identificaron 364 productos sin ingredientes, 1 sin categoría y 715 sin tamaño. Estos valores se mantienen como datos faltantes y se excluyen de los análisis que dependan de esas columnas.


## Transformación 2: eliminacion de productos sin ninguna calificacion
Se quitan/excluyen los productos de rating 0, porque en realidad no se tratan de calificaciones de cero sino a productos
que nadie calificó todavía, conservarlos significaria bajar el promedio de rating y afecta la realidad del dato.

In [38]:
print("productos con rating 0: ", (df["rating"] == 0).sum())

productos con rating 0:  398
